# 🛡️ Guardian AI — Full Pipeline on Colab T4 GPU

**Uses your laptop's mic** through the browser → Whisper transcription → MuRIL classifier → Threat scoring → Dashboard

**Requirements:** Set runtime to **T4 GPU** (Runtime → Change runtime type → T4 GPU)

## 1️⃣ Setup: Clone Repo & Install Dependencies

In [3]:
# Clone Guardian AI repo and install all dependencies
!rm -rf GuardianAi  # remove old clone if exists
!git clone https://github.com/i-shashwat-dubey/GuardianAi.git

# Install required packages
!pip install -q transformers torch faster-whisper numpy

# Add repo to Python path
import sys
sys.path.append('/content/GuardianAi')
print('✅ Setup complete!')

Cloning into 'GuardianAi'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 13 (delta 2), reused 13 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 21.68 KiB | 10.84 MiB/s, done.
Resolving deltas: 100% (2/2), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 22.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 20.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 104.1 MB/s eta 0:00:0000:010:01
✅ Setup complete!


## 2️⃣ Load Models onto T4 GPU

In [5]:
# ── Load Whisper (Speech-to-Text) on GPU ──────────────────
from faster_whisper import WhisperModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print('\nLoading Whisper medium (multilingual)...')
whisper_model = WhisperModel(
    'medium',           # multilingual: Hindi, English, Hinglish, etc.
    device=device,
    compute_type='float16' if device == 'cuda' else 'int8',
)
print('✅ Whisper loaded on GPU!')

Device: cuda
GPU: Tesla T4
VRAM: 14.6 GB

Loading Whisper medium (multilingual)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✅ Whisper loaded on GPU!


In [6]:
# ── Load MuRIL (Safety Classifier) on GPU ─────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print('Loading MuRIL (Indian languages classifier)...')
muril_tokenizer = AutoTokenizer.from_pretrained('google/muril-base-cased')
muril_model = AutoModelForSequenceClassification.from_pretrained(
    'google/muril-base-cased',
    num_labels=2,
    ignore_mismatched_sizes=True,
)
muril_model = muril_model.to(device)
muril_model.eval()
print('✅ MuRIL loaded on GPU!')
print(f'\nBoth models on {device} — ready for real-time inference!')

Loading MuRIL (Indian languages classifier)...


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

✅ MuRIL loaded on GPU!

Both models on cuda — ready for real-time inference!


## 3️⃣ Initialize Brain Components

In [7]:
# ── Initialize Threat Scorer + Memory ──────────────────────
from threat_scorer import ThreatScorer, ThreatState
from memory import MemoryManager

# Mock classifier (keyword-based) until MuRIL is fine-tuned
# We ALSO run MuRIL to show its raw output, but use mock for scoring
EMERGENCY_KEYWORDS = [
    'help', 'help me', 'save me', 'bachao', 'please help',
    'let me go', 'leave me alone', 'stop it', 'chhodo',
    'don\'t touch', 'stop touching', 'get away', 'dur raho',
    'following me', 'following', 'stalking', 'peecha',
    'kill', 'hurt', 'hit me', 'attack', 'marna', 'maarunga',
    'scared', 'afraid', 'terrified', 'dar', 'darr',
    'touching me', 'molest', 'assault', 'harass',
    'trapped', 'locked', 'kidnap', 'banda', 'aadmi',
    'scream', 'screaming', 'chillao', 'police',
    'rape', 'grab', 'pakad',
]
SAFE_KEYWORDS = [
    'movie', 'game', 'show', 'song', 'book', 'accha',
    'joke', 'funny', 'laugh', 'playing', 'mazaa',
    'grocery', 'weather', 'lunch', 'dinner', 'khana',
    'coffee', 'friend', 'happy', 'good', 'theek',
]

def mock_classify(text):
    text_lower = text.lower()
    score = 0.05
    for kw in EMERGENCY_KEYWORDS:
        if kw in text_lower:
            score += 0.25
    for kw in SAFE_KEYWORDS:
        if kw in text_lower:
            score -= 0.15
    return max(0.0, min(1.0, score))

scorer = ThreatScorer(alpha=0.5, decay_rate=0.95, panic_threshold=0.95)
memory = MemoryManager(short_term_max=20, long_term_threat_threshold=0.4)

print('✅ Brain initialized: Threat Scorer + Memory + Mock Classifier')

✅ Brain initialized: Threat Scorer + Memory + Mock Classifier


## 4️⃣ Browser Mic Capture Function

This uses JavaScript to access your **laptop's microphone** through the browser.

You'll see a mic permission popup — click **Allow**.

In [8]:
# ── JavaScript-based mic capture from browser ──────────────
from google.colab import output
from IPython.display import display, Javascript, HTML
from base64 import b64decode
import io
import numpy as np
import wave

# JavaScript code that records audio from the browser mic
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

const recordAudio = async (seconds) => {
  // Request mic access from the browser
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const mediaRecorder = new MediaRecorder(stream, {mimeType: 'audio/webm'});
  const audioChunks = [];

  mediaRecorder.addEventListener('dataavailable', event => {
    audioChunks.push(event.data);
  });

  // Start recording
  mediaRecorder.start();
  document.querySelector('#recording-status').textContent = '🔴 Recording...';

  // Record for specified seconds
  await sleep(seconds * 1000);

  // Stop recording
  const stopped = new Promise(resolve => {
    mediaRecorder.addEventListener('stop', resolve);
  });
  mediaRecorder.stop();
  stream.getTracks().forEach(track => track.stop());
  await stopped;

  document.querySelector('#recording-status').textContent = '⏳ Processing...';

  // Convert to base64
  const blob = new Blob(audioChunks, {type: 'audio/webm'});
  const reader = new FileReader();
  const readPromise = new Promise(resolve => {
    reader.addEventListener('loadend', () => resolve(reader.result));
  });
  reader.readAsDataURL(blob);
  const result = await readPromise;

  // Return base64 audio data
  return result;
};
"""

def record_audio_from_browser(seconds=5):
    """
    Record audio from your laptop mic through the Colab browser.
    Returns the audio as a temporary .webm file path.
    """
    display(HTML('<div id="recording-status" style="font-size:18px; padding:10px;">⏳ Starting...</div>'))

    # Run JavaScript to capture mic audio
    js_code = RECORD_JS + f'const result = await recordAudio({seconds}); result;'
    data = output.eval_js(js_code)

    # data is a base64 data URL: "data:audio/webm;base64,XXXXXXX"
    # Extract the base64 part
    b64_data = data.split(',')[1]
    audio_bytes = b64decode(b64_data)

    # Save as temp file for Whisper
    temp_path = '/tmp/guardian_mic_input.webm'
    with open(temp_path, 'wb') as f:
        f.write(audio_bytes)

    return temp_path

print('✅ Mic capture function ready!')
print('   When you run the pipeline, your browser will ask for mic permission.')

✅ Mic capture function ready!
   When you run the pipeline, your browser will ask for mic permission.


## 5️⃣ Process Pipeline Function

In [9]:
# ── Full processing pipeline ──────────────────────────────
import time

utterance_count = 0

def process_audio(audio_path):
    """
    Full pipeline: audio file → transcribe → classify → score → dashboard.
    """
    global utterance_count

    # ── Step 1: Transcribe with Whisper (GPU) ──────────────
    t0 = time.time()
    segments, info = whisper_model.transcribe(audio_path, beam_size=1)
    text_parts = [seg.text.strip() for seg in segments if seg.text.strip()]
    transcription_time = time.time() - t0

    if not text_parts:
        print('  🔇 No speech detected in this recording.')
        return

    text = ' '.join(text_parts)
    utterance_count += 1

    # ── Step 2: Classify with MuRIL (GPU) ─────────────────
    t1 = time.time()
    context = memory.get_context_for_classifier()
    classifier_input = f'{context} {text}' if context else text

    inputs = muril_tokenizer(
        classifier_input,
        padding='max_length', truncation=True,
        max_length=480, return_tensors='pt',
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    token_count = (inputs['attention_mask'] == 1).sum().item()

    with torch.no_grad():
        outputs = muril_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    muril_emergency = probs[0][1].item()
    muril_safe = probs[0][0].item()
    classify_time = time.time() - t1

    # ── Step 3: Mock classifier score (for actual scoring) ─
    mock_score = mock_classify(text)

    # ── Step 4: Update memory + threat scorer ─────────────
    memory.add_utterance(text, mock_score)
    state = scorer.update(mock_score)

    # ── Step 5: Display dashboard ─────────────────────────
    state_colors = {
        ThreatState.SAFE: '🟢', ThreatState.MONITOR: '🟡',
        ThreatState.ALERT: '🟠', ThreatState.EMERGENCY: '🔴',
    }

    bar_len = 30
    filled = int(scorer.threat_level * bar_len)
    bar = '▓' * filled + '░' * (bar_len - filled)

    lang = info.language if hasattr(info, 'language') else '??'

    print(f'\n{"═" * 65}')
    print(f'  🎙  UTTERANCE #{utterance_count}  (lang: {lang})')
    print(f'{"─" * 65}')
    print(f'  Speech:    {text}')
    print(f'  Whisper:   {transcription_time:.2f}s  |  MuRIL: {classify_time:.3f}s  |  Tokens: {token_count}')
    print(f'{"─" * 65}')
    print(f'  MuRIL raw: SAFE={muril_safe:.1%}  EMERGENCY={muril_emergency:.1%}  (⚠️ not fine-tuned)')
    print(f'  Mock score: {mock_score:.2f}')
    print(f'{"─" * 65}')
    print(f'  Threat:    {bar} {scorer.threat_level:.2f}')
    print(f'  State:     {state_colors.get(state, "")} {state.value}')
    print(f'{"─" * 65}')
    print(f'  Short-term: {len(memory.short_term)} utterances  |  Long-term: {len(memory.long_term)} flagged')
    if memory.long_term:
        for u in memory.long_term[-3:]:
            print(f'    ⚠ [{u.formatted_time()}] ({u.threat_score:.2f}) {u.text[:50]}')
    if state in (ThreatState.ALERT, ThreatState.EMERGENCY):
        print(f'  \n  🚨🚨🚨 {state.value} — EMERGENCY ACTIONS WOULD TRIGGER 🚨🚨🚨')
    print(f'{"═" * 65}')

print('✅ Pipeline function ready!')

✅ Pipeline function ready!


## 6️⃣ 🎙 Run Guardian AI!

Run this cell to start listening. Each run records **5 seconds** of audio from your laptop mic.

**Re-run this cell** to record another segment. It accumulates memory across runs.

In [10]:
# 🎙 Record 5 seconds from your laptop mic and process
RECORD_SECONDS = 5

print(f'🎙 Recording {RECORD_SECONDS} seconds from your mic...')
print('   (Allow mic access if your browser asks)\n')

audio_path = record_audio_from_browser(seconds=RECORD_SECONDS)
process_audio(audio_path)

🎙 Recording 5 seconds from your mic...
   (Allow mic access if your browser asks)



KeyboardInterrupt: 

## 7️⃣ 🔄 Continuous Loop Mode (Optional)

Run this to continuously record and process in a loop.

**Stop with the ⏹ button** or Runtime → Interrupt execution.

In [ ]:
# 🔄 Continuous listening mode — press STOP to end
RECORD_SECONDS = 5
NUM_ROUNDS = 20  # max rounds before auto-stop

print('🛡️ GUARDIAN AI — CONTINUOUS MONITORING')
print(f'   Recording {RECORD_SECONDS}s segments, up to {NUM_ROUNDS} rounds')
print('   Press ⏹ to stop\n')

for i in range(NUM_ROUNDS):
    try:
        print(f'\n--- Round {i+1}/{NUM_ROUNDS} ---')
        audio_path = record_audio_from_browser(seconds=RECORD_SECONDS)
        process_audio(audio_path)
    except KeyboardInterrupt:
        print('\n⏹ Stopped by user.')
        break
    except Exception as e:
        print(f'⚠ Error: {e}')
        break

# Session summary
print(f'\n{"═" * 65}')
print(f'  🛡️ SESSION SUMMARY')
print(f'  Final state: {scorer.current_state.value}')
print(f'  Final threat: {scorer.threat_level:.3f}')
print(f'  Utterances: {len(memory.short_term)} short-term, {len(memory.long_term)} flagged')
if memory.long_term:
    print(f'  ⚠ Flagged utterances:')
    for u in memory.long_term:
        print(f'    [{u.formatted_time()}] ({u.threat_score:.2f}) {u.text}')
print(f'{"═" * 65}')